<a href="https://colab.research.google.com/github/BeachBall2024/GEO-spa-algo/blob/main/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Workflow and Introduction
Study:
1. Load data and basic functions
- Lea
2. preprocess Flickr data for typos, upper-lower case
3. construct sound wheel
- Julia
4. divide streets into segments (name, then again for a max length)
- Muriel
5. create Buffer around streets
- Guillermo
6. check which points lay inside buffer and match them to sound categories, count

In [ ]:
#Import gihub repo libraries
!git clone https://github.com/BeachBall2024/GEO-spa-algo.git
%cd GEO-spa-algo

2. Import Python packages

In [ ]:
#importing important packages

3. Import basic geometric functions
(Functions are dividet in different classes based on shape)

- Point
- Line
- Polygon

In [ ]:
#POINT
class Point():
    # initialise
    def __init__(self, x=None, y=None):
        self.x = x
        self.y = y

    # representation
    def __repr__(self):
        return f'Point(x={self.x}, y={self.y})'

    # Test for equality between Points
    def __eq__(self, other):
        if not isinstance(other, Point): #test if point is within the class
            # don't attempt to compare against unrelated types
            return NotImplemented
        return self.x == other.x and self.y == other.y

    # We need this method so that the class will behave sensibly in sets and dictionaries
    def __hash__(self):
        return hash((self.x, self.y))

class PointGroup():
    # initialise
    def __init__(self, data=None, xcol=None, ycol=None):
        self.points = []
        self.size = len(data)
        for d in data:
            self.points.append(Point(d[xcol], d[ycol]))

    # representation
    def __repr__(self):
        return f'PointGroup containing {self.size} points'

    # create index of points in group for referencing
    def __getitem__(self, key):
        return self.points[key]

In [ ]:
#LINE
class Segment():
    # initialise
    def __init__(self, p0, p1, sid=None):
        self.start = p0
        self.end = p1
        self.sid = sid

    # representation
    def __repr__(self):
        return f'Segment with start {self.start} and end {self.end}.'

    # test if segment is identical to another segment (using Point method isEqual)
    def isIdentical(self, other):
        if (self.start.isEqual(other.start)
            or self.start.isEqual(other.end)) and (self.end.isEqual(other.end)
                                                   or self.end.isEqual(other.start)):
            return True
        else:
            return False

    # determine if intersects with another segment (using Point method sideLine)
    # - should we incorporate testing for identical segments and non-zero lengths?
    def intersects(self, other):
        # create bounding boxes for each segment, using Bbox class
        self_bbox = Bbox(self)
        other_bbox = Bbox(other)

        # test if bounding boxes overlap (using Bbox method testOverlap)
        bbox_overlap = self_bbox.testOverlap(other_bbox)

        # if the two bboxes do not overlap, there can be no intersection
        if bbox_overlap == False:
            return False

        else:
            return True

    def length(self):
        #calculate the distance of the segment

In [ ]:
#POLYGON
# Polygon class for polygons, assumes initial data is in a spatially sorted order
class PointGroup:
    # initialise
    def __init__(self, data=None, xcol=None, ycol=None):
        self.points = []
        self.size = len(data)
        for d in data:
            self.points.append(Point(d[xcol], d[ycol]))

    # representation
    def __repr__(self):
        return f'Polygon PointGroup containing {self.size} points'

    # test if polygon is closed: first and last point should be identical
    def isClosed(self):
        start = self.points[0]
        end = self.points[-1]
        return start == end

    def removeDuplicates(self):
        oldn = len(self.points)
        self.points = list(dict.fromkeys(self.points)) # Get rid of the duplicates
        self.points.append(self.points[0]) # Our polygon must have one duplicate - we put it back now
        n = len(self.points)
        self.size = n   # see how the absence/presence of this line makes changes
        print(f'The old polygon had {oldn} points, now we only have {n}.')

# 4. Data Loading

  We load three datasets:
- **Flickr Images**
- **Sound Wheel**
- **Streets of Switzerland**


In [ ]:
#import data

# 5. Preprocessing of Flickr Images



In [ ]:
#preprocess of Flickr Images

6. Division of Streets into Segments

In [30]:
import math
import pathlib
import pandas as pd

# ─── configuration ────────────────────────────────────────────────────────────
INPUT_FILE  = "content/amtliches-strassenverzeichnis_ch_2056.csv"
OUT_SEGS    = "zurich_street_segments.csv"
OUT_SUMMARY = "zurich_street_segments_summary.csv"

MAX_SEGMENT_LENGTH_M = 500      # split threshold in metres

# "city"   → COM_NAME == "Zürich"   (city only)
# "canton" → COM_CANTON == "ZH"     (all ZH communes – recommended for segmentation)
MODE = "canton"
# ──────────────────────────────────────────────────────────────────────────────


def euclidean_dist(e1, n1, e2, n2):
    """Straight-line distance in metres between two LV95 points."""
    return math.sqrt((e2 - e1) ** 2 + (n2 - n1) ** 2)


def sort_points_nearest_neighbour(pts):
    """
    Order a list of (easting, northing, meta_dict) tuples using a greedy
    nearest-neighbour chain starting from the westernmost point.
    Gives a reasonable linear ordering for streets spread across communes.
    """
    if len(pts) == 1:
        return pts

    pts = list(pts)  # copy so we can pop
    start = min(range(len(pts)), key=lambda i: pts[i][0])  # lowest easting
    ordered = [pts.pop(start)]

    while pts:
        le, ln, _ = ordered[-1]
        nearest = min(range(len(pts)),
                      key=lambda i: euclidean_dist(le, ln, pts[i][0], pts[i][1]))
        ordered.append(pts.pop(nearest))

    return ordered


def build_segments(sorted_pts, max_len):
    """
    Split an ordered list of points into segments whose cumulative length
    does not exceed max_len metres.

    Returns a list of dicts, one per segment.
    """
    segments = []
    cur_pts  = [sorted_pts[0]]
    cur_len  = 0.0

    for prev, curr in zip(sorted_pts, sorted_pts[1:]):
        e1, n1, _ = prev
        e2, n2, _ = curr
        d = euclidean_dist(e1, n1, e2, n2)

        if cur_len + d > max_len and len(cur_pts) >= 1:
            # flush current segment, start new one from the boundary point
            segments.append(_pack_segment(cur_pts, cur_len, len(segments) + 1))
            cur_pts = [prev, curr]
            cur_len = d
        else:
            cur_pts.append(curr)
            cur_len += d

    if cur_pts:
        segments.append(_pack_segment(cur_pts, cur_len, len(segments) + 1))

    return segments


def _pack_segment(pts, length_m, seg_id):
    e0, n0, _ = pts[0]
    e1, n1, _ = pts[-1]
    return {
        "segment_id":        seg_id,
        "point_count":       len(pts),
        "length_m":          round(length_m, 2),
        "start_easting":     round(e0, 3),
        "start_northing":    round(n0, 3),
        "end_easting":       round(e1, 3),
        "end_northing":    round(n1, 3),
        "midpoint_easting":  round((e0 + e1) / 2, 3),
        "midpoint_northing": round((n0 + n1) / 2, 3),
        "esids":             ",".join(str(p[2]["STR_ESID"])      for p in pts),
        "communes":          ",".join(sorted({p[2]["COM_NAME"]   for p in pts})),
        "zip_labels":        ",".join(sorted({p[2]["ZIP_LABEL"]  for p in pts})),
    }


# ─── main ─────────────────────────────────────────────────────────────────────
def main():
    script_dir = pathlib.Path('.') # Use current directory since we cd'd into the repo

    # ── 1. Load ──────────────────────────────────────────────────────────────
    print(f"Loading {INPUT_FILE} ...")
    df = pd.read_csv(
        script_dir / INPUT_FILE,
        sep=";",
        encoding="utf-8-sig",
        low_memory=False,
    )
    print(f"  {len(df):,} total rows loaded.")

    # ── 2. Filter ────────────────────────────────────────────────────────────
    if MODE == "city":
        subset = df[df["COM_NAME"] == "Zürich"].copy()
        label  = "Zürich city"
    elif MODE == "canton":
        subset = df[df["COM_CANTON"] == "ZH"].copy()
        label  = "Canton Zürich (ZH)"
    else:
        raise ValueError(f"Unknown MODE: {MODE!r}. Use 'city' or 'canton'.")

    subset = subset.dropna(subset=["STR_EASTING", "STR_NORTHING"])
    print(f"  {len(subset):,} rows for {label}.")

    # ── 3. Group by name and segment ─────────────────────────────────────────
    street_names = sorted(subset["STN_LABEL"].unique())
    print(f"  Processing {len(street_names):,} unique street names ...")

    segment_rows = []
    summary_rows = []

    for name in street_names:
        group = subset[subset["STN_LABEL"] == name]
        pts   = [(r["STR_EASTING"], r["STR_NORTHING"], r.to_dict())
                 for _, r in group.iterrows()]
        pts   = sort_points_nearest_neighbour(pts)
        segs  = build_segments(pts, MAX_SEGMENT_LENGTH_M)

        total_len = sum(s["length_m"] for s in segs)

        for seg in segs:
            segment_rows.append({"street_name": name, **seg})

        summary_rows.append({
            "street_name":    name,
            "total_points":   len(pts),
            "num_segments":   len(segs),
            "total_length_m": round(total_len, 2),
        })

    # ── 4. Save ───────────────────────────────────────────────────────────────
    seg_df = pd.DataFrame(segment_rows)
    sum_df = pd.DataFrame(summary_rows)

    seg_out = script_dir / OUT_SEGS
    sum_out = script_dir / OUT_SUMMARY

    seg_df.to_csv(seg_out, index=False, encoding="utf-8-sig")
    sum_df.to_csv(sum_out, index=False, encoding="utf-8-sig")

    print(f"\nDone.")
    print(f"  Segments file : {seg_out}  ({len(seg_df):,} rows)")
    print(f"  Summary file  : {sum_out}  ({len(sum_df):,} rows)")

    # ── 5. Quick stats ────────────────────────────────────────────────────────
    split = sum_df[sum_df["num_segments"] > 1]
    print(f"\n  Streets with >1 segment : {len(split):,}")
    if len(split) > 0:
        worst = sum_df.loc[sum_df["num_segments"].idxmax()]
        print(f"  Most-split street       : {worst['street_name']}  "
              f"({worst['num_segments']} segments, {worst['total_length_m']:.0f} m)")
        print(f"\n  Top 10 longest streets (estimated by point spread):")
        top10 = sum_df.nlargest(10, "total_length_m")[
            ["street_name", "total_points", "num_segments", "total_length_m"]
        ]
        print(top10.to_string(index=False))


if __name__ == "__main__":
    main()

Loading content/amtliches-strassenverzeichnis_ch_2056.csv ...


FileNotFoundError: [Errno 2] No such file or directory: 'content/amtliches-strassenverzeichnis_ch_2056.csv'

In [27]:
import os

print("Listing files in the current directory (GEO-spa-algo):")
for root, dirs, files in os.walk('.'):
    level = root.replace('./', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Listing files in the current directory (GEO-spa-algo):
./
.config/
    .last_survey_prompt.yaml
    .last_update_check.json
    default_configs.db
    hidden_gcloud_config_universe_descriptor_data_cache_configs.db
    .last_opt_in_prompt.yaml
    active_config
    config_sentinel
    gce
    logs/
        2026.04.16/
            13.27.28.140888.log
            13.28.06.952902.log
            13.28.22.950479.log
            13.27.53.684484.log
            13.28.21.827521.log
            13.28.05.189231.log
    configurations/
        config_default
sample_data/
    anscombe.json
    README.md
    mnist_train_small.csv
    california_housing_test.csv
    california_housing_train.csv
    mnist_test.csv


# 7. Creation of Buffer

In [ ]:
# create Buffer

8. Point in Polygon


In [ ]:
#point in buffer testing

9. Visualization

In [ ]:
#sound map